# Group 16 | V3 Acceptance on Tesla T4

Notebook nghiệm thu P8-T04 cho sáu cặp synthetic và 12 cấu hình ảnh thật. Các đề mục bên dưới đi từ xác minh nguồn, kiểm tra hình học, CLI và CSV đến kiểm tra bộ nhớ. Kết quả challenge được xem riêng theo trạng thái và chất lượng ảnh.

## Quick Start and Required Inputs

1. Mở notebook trong Google Colab, chọn **Runtime → Change runtime type → T4 GPU**.
2. Chờ T01/T02 bàn giao commit production. Nhánh dùng để chạy phải chứa commit production và các commit kiểm thử P8-T04. Điền SHA production đủ 40 ký tự và header CSV nguyên dòng do Thành xác nhận. Chốt ref hoặc SHA của checkout kiểm thử trong ô cấu hình.
3. Để mặc định, notebook lấy dataset cuối kỳ từ Drive và lấy ảnh nguồn P7 từ Drive để tạo sáu cặp synthetic. Nếu Drive chặn tải hoặc file sinh lại lệch hash, điền đường dẫn ZIP hoặc thư mục đã có trong Colab vào ô cấu hình. Notebook kiểm tra toàn bộ hash trước khi chạy.
4. Chọn **Runtime → Run all**, xem gallery challenge rồi ghi nhận xét ở ô cuối. Chạy lại ô tổng hợp và tải gói bằng chứng về để đưa vào PR.

In [ ]:
# @title Verification Settings { display-mode: "form" }
GIT_URL = "https://github.com/nh-atuan/panorama-stitching.git"
VERIFICATION_REF = "refs/heads/chuong/p8-v3-verification"  # @param {type:"string"}
PRODUCTION_SHA = ""  # @param {type:"string"}
T02_CSV_HEADER = ""  # @param {type:"string"}
DATASET_DRIVE_URL = "https://drive.google.com/drive/folders/1_J_q5q-K1g7ZNnczTWvXBzwpOsof9ghL?usp=sharing"  # @param {type:"string"}
DATASET_INPUT = ""  # @param {type:"string"}
SYNTHETIC_INPUT = ""  # @param {type:"string"}
SYNTHETIC_SOURCE = ""  # @param {type:"string"}

## Source and Runtime Preflight

Notebook kiểm tra checkout đã tích hợp T01/T02, xác nhận source chương trình và script build khớp production SHA, rồi lưu cấu hình T4 và CUDA. Điền các giá trị còn trống trước khi bật GPU để tiết kiệm hạn mức Colab.

In [ ]:
# @title Resolve Source and Capture CUDA Environment
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
import csv
import json
import os
import re
import shutil
import stat
import subprocess
import sys
import tempfile
import zipfile

if not re.fullmatch(r"[0-9a-f]{40}", PRODUCTION_SHA):
    raise ValueError("BLOCKED: enter the full V3 production SHA supplied by T01/T02.")
if not T02_CSV_HEADER or len(next(csv.reader([T02_CSV_HEADER]))) != 11:
    raise ValueError("BLOCKED: enter the 11-column CSV header agreed with T02.")
from IPython.display import Markdown, display

WORK = Path(tempfile.mkdtemp(prefix="group16_p8_", dir="/content"))
EVIDENCE = WORK / "evidence"
LOGS = EVIDENCE / "logs"
LOGS.mkdir(parents=True)
PROJECT = WORK / "source"

def output(argv, cwd=None):
    return subprocess.check_output(list(map(str, argv)), cwd=cwd, text=True, timeout=120).strip()

subprocess.run(["git", "clone", "--quiet", "--no-checkout", GIT_URL, str(PROJECT)], check=True, timeout=240)
subprocess.run(["git", "fetch", "--quiet", "origin", VERIFICATION_REF], cwd=PROJECT, check=True, timeout=120)
subprocess.run(["git", "checkout", "--detach", "--quiet", "FETCH_HEAD"], cwd=PROJECT, check=True, timeout=120)
VERIFY_SHA = output(["git", "rev-parse", "HEAD"], PROJECT)
for required in (PRODUCTION_SHA, "36119b0a72b46dd1dce4e39b4484bc44d00e063f"):
    if subprocess.run(["git", "merge-base", "--is-ancestor", required, VERIFY_SHA], cwd=PROJECT).returncode:
        raise ValueError(f"BLOCKED: checkout {VERIFY_SHA} does not contain {required}.")
if subprocess.run(["git", "diff", "--quiet", PRODUCTION_SHA, VERIFY_SHA, "--", "src", "scripts/build.py"],
                  cwd=PROJECT).returncode:
    raise ValueError("BLOCKED: build inputs differ from the declared production SHA.")
required_files = ("src/v3/v3_pipeline.h", "src/v3/v3_pipeline.cu", "scripts/verify_v3_cli.py",
                  "tests/gpu/test_gpu_v3_pipeline.cu")
missing = [name for name in required_files if not (PROJECT / name).is_file()]
if missing:
    raise ValueError(f"BLOCKED: missing integrated files: {missing}")

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "scripts"))
sys.path.insert(0, str(PROJECT / "tests"))
from profiling_common import capture, inspect_environment, write_json
from run_benchmark import sha256, version_columns
from run_tests import synthetic_arguments
from verify_v2_dataset import source_snapshot

if next(csv.reader([T02_CSV_HEADER])) != version_columns(3):
    raise ValueError("T02 header differs from version_columns(3); resolve the contract before acceptance.")

def recorded(label, argv, timeout=600):
    result, stdout, stderr = capture(argv, LOGS / label, timeout=timeout)
    print(f"{label}: exit={result['returncode']}; logs/{label}.*")
    return result, stdout, stderr

def checked(label, argv, timeout=600):
    result, stdout, stderr = recorded(label, argv, timeout)
    if result["returncode"] != 0:
        raise RuntimeError(f"{label} failed: {stderr[-1500:]}")
    return stdout

ENV = inspect_environment(require_t4=True)
gpu = checked("gpu", ["nvidia-smi", "--query-gpu=index,name,compute_cap,driver_version", "--format=csv,noheader"])
if not gpu.splitlines() or any(row[2].strip() != "7.5" for row in csv.reader(gpu.splitlines())):
    raise ValueError(f"Expected compute capability 7.5: {gpu}")
ENV["gpu_details"] = gpu.strip()
ENV["nvcc"] = checked("nvcc", ["nvcc", "--version"]).strip()
ENV["sanitizer"] = checked("sanitizer_version", ["compute-sanitizer", "--version"]).strip()
ENV["sanitizer_help"] = "logs/sanitizer_help.stdout"
san_help = checked("sanitizer_help", ["compute-sanitizer", "--tool", "memcheck", "--help"])
if not all(flag in san_help for flag in ("--leak-check", "--error-exitcode")):
    raise ValueError("Installed Compute Sanitizer does not advertise the required options.")
probe = WORK / "cuda_runtime_probe.cu"
probe.write_text(
    '#include <cuda_runtime_api.h>\n#include <cstdio>\n'
    'int main() { int v = 0; if (cudaRuntimeGetVersion(&v) != cudaSuccess) return 1; '
    'std::printf("%d\\n", v); return 0; }\n',
    encoding="utf-8",
)
checked("runtime_build", ["nvcc", "-arch=sm_75", probe, "-o", WORK / "cuda_runtime_probe"])
ENV["cuda_runtime_version_encoded"] = int(checked("runtime_version", [WORK / "cuda_runtime_probe"]).strip())
ENV["build_configuration"] = "scripts/build.py, nvcc, sm_75, C++11, -O2, --fmad=false"
ENV["git"] = {"ref": VERIFICATION_REF, "remote": GIT_URL, "checkout_sha": VERIFY_SHA, "production_sha": PRODUCTION_SHA,
              "source_snapshot": source_snapshot()["snapshot_sha256"]}
ENV["started_utc"] = datetime.now(timezone.utc).isoformat()
write_json(EVIDENCE / "environment.json", ENV)
print(json.dumps(ENV, indent=2))

## Dataset and Synthetic Fixture Integrity

Notebook ưu tiên tải dataset đã đóng gói từ thư mục Drive cuối kỳ. Nếu không lấy được gói, notebook thử quy trình tải ảnh gốc và chuẩn bị dataset của P7. Synthetic dùng ảnh nguồn P7 đã xác minh. Có thể chỉ định ZIP hoặc thư mục giải nén sẵn bằng đường dẫn Colab, kể cả đường dẫn trên Drive đã mount.

In [ ]:
P7_MANIFEST_SHA256 = '99fc712988075326a351d0bf580405e418bb28d24d56d4cc1b31505c480bdd9d'
P7_SYNTHETIC_SHA256 = {
    "translation/small_left.png": "e99af21cf5b2bde2e83a7ebe0f653bc1a015df5a7a56bc5d1dcdc1813f5c7d79",
    "translation/small_right.png": "6161099d9e4ed19580c284452143f08e704eccc6f45cbea882d9a1d67747a0d0",
    "translation/small_truth.json": "d3bb8ffd85dde153bc01be3de96ea5d08c74b065b0f68cd51bf4b5896fcfca15",
    "translation/medium_left.png": "d4759d92ce2bf87601c7a94ce9e2c9b8ad9a088f6ca3ced7b1811d31dc5fd6aa",
    "translation/medium_right.png": "22dbbb85fe82116196bf372b152157f88f75a112e2c72ba7f22e56fb9ca11056",
    "translation/medium_truth.json": "9f708fadb7d1c0942b2e8bb7166b67642d2ccc4bdc4b343a1ba0d75c616586c1",
    "translation/large_left.png": "872abfd3f1a42d4506963bbb9db91fa82f6fe67c4711723d35764a3055f5e5de",
    "translation/large_right.png": "cf5c46aa8aca4b32af46c6243690bb8de8b70f3f21ca02aed63bc91a1b7c5894",
    "translation/large_truth.json": "7075d5415aba31ff3dbfffe55dec81b3849024c44bee474aebb14eac9671dada",
    "projective/small_left.png": "8debea84d29cc27333d6d0f8a2f4a8775c34ad384231c2f6d6153966daadadc4",
    "projective/small_right.png": "a8318b868154fda0e2b2a49c343cdca2eb5c73746b4990ab99947b7ed13b8cc6",
    "projective/small_truth.json": "cf2d4505a02c306c9990a0594dc966154ece5966279032a252c61d8649bd266a",
    "projective/medium_left.png": "75e138e097006eecee60472ae832d8d741dedd7175fcf8bfbe584bc164f3726f",
    "projective/medium_right.png": "c6c8c52f1a1eacc6b89d4ced642fcf1cd8760b517fb4264323909dd355bbe0a9",
    "projective/medium_truth.json": "e489bd356653b78706514ecdf3535ccbce0d7d7522ec1106ba004551bf9e8332",
    "projective/large_left.png": "78964b98ea58b04a5ee7cd470ed9ec57396f7e505b30f0bf44beb22452007174",
    "projective/large_right.png": "37786a3336c48779388da705a7466d642e8230151d1f69ed2dd7080d48182fe6",
    "projective/large_truth.json": "63ba932a8e6668804eec44c3535b07207b8e52ab1bdc6f8233a1e8eee963cc61"
}

from fetch_and_prepare_dataset import ensure_gdown
from verify_v2_dataset import official_inputs

def locate_input(raw_path, marker, label):
    path = Path(raw_path)
    if path.is_dir():
        root = path
    elif path.is_file():
        root = WORK / label
        root.mkdir()
        with zipfile.ZipFile(path) as archive:
            for member in archive.infolist():
                name = PurePosixPath(member.filename)
                if (name.is_absolute() or ".." in name.parts or "\\" in member.filename
                        or stat.S_ISLNK(member.external_attr >> 16)):
                    raise ValueError(f"Unsafe ZIP member: {member.filename}")
            archive.extractall(root)
    else:
        raise FileNotFoundError(f"BLOCKED: upload or configure {label}: {path}")
    parts = PurePosixPath(marker).parts
    matches = [match for match in root.rglob(parts[-1]) if match.parts[-len(parts):] == parts]
    if len(matches) != 1:
        raise ValueError(f"Expected one {marker} in {root}, got {len(matches)}")
    return matches[0].parents[len(parts) - 1]

P7_SYNTHETIC_SOURCE_SHA256 = "e9fbe7ecfed6344a36610fe50f2c6019bcd1525f4e217c033ed8ded4f90bf1e8"
RAW = WORK / "raw"

def verify_dataset(dataset):
    if sha256(dataset / "manifest.csv") != P7_MANIFEST_SHA256:
        raise ValueError("Dataset manifest differs from the P7 evidence manifest.")
    return official_inputs(dataset)

def fetch_dataset():
    if DATASET_INPUT:
        dataset = locate_input(DATASET_INPUT, "manifest.csv", "dataset_input")
        return dataset, verify_dataset(dataset), "configured input"

    try:
        ensure_gdown()
        import gdown
        downloaded = WORK / "dataset_drive"
        downloaded.mkdir()
        gdown.download_folder(url=DATASET_DRIVE_URL, output=str(downloaded) + "/", use_cookies=False)
        archives = list(downloaded.rglob("dataset.zip"))
        if len(archives) != 1:
            raise ValueError(f"Expected one dataset.zip on Drive, found {len(archives)}")
        dataset = locate_input(archives[0], "manifest.csv", "dataset_drive_extract")
        return dataset, verify_dataset(dataset), "official Drive ZIP"
    except Exception as error:
        print(f"Official Drive ZIP unavailable or invalid: {error}")
        print("Trying the P7 raw-photo preparation workflow.")

    dataset = WORK / "dataset_from_raw"
    try:
        checked("prepare_dataset", [sys.executable, "scripts/fetch_and_prepare_dataset.py",
                                    "--raw", RAW, "--out", dataset], 1200)
        return dataset, verify_dataset(dataset), "P7 raw-photo workflow"
    except (OSError, RuntimeError, ValueError) as error:
        raise RuntimeError(
            "BLOCKED: automatic dataset preparation did not reproduce P7. "
            "Set DATASET_INPUT to the official dataset.zip or its extracted directory."
        ) from error

DATASET, CASES, DATASET_ORIGIN = fetch_dataset()
if SYNTHETIC_INPUT:
    SYNTHETIC = locate_input(SYNTHETIC_INPUT, "translation/small_truth.json", "synthetic_input")
    SYNTHETIC_ORIGIN = "configured fixtures"
else:
    source = Path(SYNTHETIC_SOURCE) if SYNTHETIC_SOURCE else RAW / "tuan/standard1_left.jpg"
    if not SYNTHETIC_SOURCE and not source.is_file():
        from fetch_and_prepare_dataset import download_raw
        try:
            ensure_gdown()
            download_raw(RAW)
        except Exception as error:
            raise RuntimeError("BLOCKED: cannot fetch the P7 synthetic source from Drive. "
                               "Set SYNTHETIC_INPUT to the verified P7 fixtures ZIP.") from error
    if not source.is_file() or sha256(source) != P7_SYNTHETIC_SOURCE_SHA256:
        raise ValueError("The synthetic source does not match P7. "
                         "Set SYNTHETIC_INPUT to the verified P7 fixtures ZIP.")
    SYNTHETIC = WORK / "synthetic_from_source"
    checked("prepare_synthetic", [sys.executable, "scripts/make_synthetic_pair.py",
                                  "--source", source, "--out", SYNTHETIC], 600)
    SYNTHETIC_ORIGIN = "configured source" if SYNTHETIC_SOURCE else "P7 raw Drive source"
print(f"Dataset: {DATASET_ORIGIN}; synthetic: {SYNTHETIC_ORIGIN}")
for name, expected in P7_SYNTHETIC_SHA256.items():
    if not (SYNTHETIC / name).is_file() or sha256(SYNTHETIC / name) != expected:
        raise ValueError(f"Synthetic fixture differs from P7: {name}. "
                         "Set SYNTHETIC_INPUT to the verified P7 fixtures ZIP.")
SYNTHETIC_ARGS = synthetic_arguments(SYNTHETIC)
shutil.copy2(DATASET / "manifest.csv", EVIDENCE / "dataset_manifest.csv")
write_json(EVIDENCE / "fixture_manifest.json", {"dataset_sha256": P7_MANIFEST_SHA256,
           "dataset_origin": DATASET_ORIGIN, "synthetic_origin": SYNTHETIC_ORIGIN,
           "synthetic_sha256": P7_SYNTHETIC_SHA256, "ransac_seed": 16, "cases": len(CASES)})
print(f"Verified {len(CASES)} real-image configurations and {len(SYNTHETIC_ARGS) // 13} synthetic pairs.")

## Build and Pipeline Regression

Runner Python của nhóm build với sm_75. Suite V3 đọc cùng input, Config và seed cho V0–V3, kiểm tra lần lượt trạng thái, H, canvas và RGB. Mốc H_gt là RMSE ≤ 0,5 px và Max ≤ 1 px trên tập Q của P7. Sai lệch chiếu giữa phiên bản dưới 0,1 px và Max RGB ≤ 1 mức mỗi kênh.

In [ ]:
# @title Build Production and Run V0–V3 Regression
from verify_v2_dataset import stitch_command
STITCH = PROJECT / "build/stitch"
SUITE = PROJECT / "build/tests_gpu/gpu_v3_pipeline"
GATES = {}
GATES["production_build"] = recorded("build", [sys.executable, "scripts/build.py", "--compiler", "nvcc",
                                      "--require-t4", "--out-dir", "build"], 1800)[0]["returncode"] == 0
if not GATES["production_build"]:
    raise RuntimeError("Build failed. Download evidence from the final cell after reviewing logs/build.stderr.")
ENV["stitch_sha256"] = sha256(STITCH)
write_json(EVIDENCE / "environment.json", ENV)

runner = [sys.executable, "tests/run_tests.py", "--compiler", "nvcc", "--require-t4"]
GATES["v0_regression"] = recorded("v0_regression", [*runner, "--group", "v0"], 1800)[0]["returncode"] == 0
gpu, gpu_stdout, _ = recorded("gpu_regression", [
    *runner, "--group", "gpu", "--suite", "gpu_pipeline", "--suite", "gpu_v2_pipeline",
    "--suite", "gpu_v3_pipeline", "--require-v2-pipeline", "--v2-synthetic-dir", SYNTHETIC,
    "--v3-synthetic-dir", SYNTHETIC, "--v3-dataset-dir", DATASET,
], 3600)
GATES["pipeline_regression"] = gpu["returncode"] == 0 and "PASS GPU V3 pipeline (full acceptance)" in gpu_stdout
if SUITE.is_file():
    ENV["suite_sha256"] = sha256(SUITE)
    write_json(EVIDENCE / "environment.json", ENV)
synthetic_names = {f"{kind}/{size}" for kind in ("translation", "projective") for size in ("small", "medium", "large")}
SYNTHETIC_RESULTS = {line.partition(":")[0]: line.partition(": ")[2] for line in gpu_stdout.splitlines()
                     if line.partition(":")[0] in synthetic_names and "V3 vs H_gt" in line}
GATES["six_synthetic"] = set(SYNTHETIC_RESULTS) == synthetic_names
write_json(EVIDENCE / "synthetic_results.json", SYNTHETIC_RESULTS)
for name, metrics in SYNTHETIC_RESULTS.items():
    print(name, metrics)

## Real-Image Matrix and CLI Contract

Mỗi cấu hình chạy qua CLI production của bốn phiên bản. So sánh H, canvas và RGB chỉ áp dụng khi cả bốn tạo được panorama. Các cột status ghi lại chính xác kết quả quan sát. Timing ở đây dùng để kiểm tra contract CSV.

Một lượt riêng chạy benchmark runner cho V3 để kiểm tra tương thích với parser. Các kết quả timing của lượt này phục vụ P8-T04.

In [ ]:
# @title Run the Production Matrix and Validate CLI Output
from verify_v2_dataset import DRIFT_PX, RGB_TOLERANCE, RUNS, projection_drift, rgb_delta, run_version
from verify_v3_cli import verify_csv

ROWS = []
VERSIONS = (0, 1, 2, 3)
for site, pair, size, width, height, left, right in CASES:
    name = f"{site}/{size}"
    folder = EVIDENCE / "panoramas" / site / size
    folder.mkdir(parents=True)
    outputs = [folder / f"v{version}.png" for version in VERSIONS]
    prefixes = [LOGS / f"{site}_{size}_v{version}" for version in VERSIONS]
    versions = [run_version(STITCH, left, right, outputs[v], v, prefixes[v]) for v in VERSIONS]
    expected = "NumericalFailure" if site == "challenge" and size != "small" else "Success"
    errors = [f"V{v}: {error}" for v in VERSIONS for error in versions[v]["validation_errors"]]
    statuses = [result["status"] for result in versions]
    row = {"case": name, "pair": pair, "expected": expected, "statuses": statuses,
           "versions": versions, "errors": errors}
    if statuses != [expected] * len(VERSIONS):
        errors.append(f"Expected {expected} for V0–V3, got {statuses}")
    elif expected != "Success":
        for v, result in enumerate(versions):
            if (result["returncode"] != 1 or "S6 warp_blend" not in result["error"]
                    or outputs[v].exists() or prefixes[v].with_suffix(".stdout").read_text().strip()):
                errors.append(f"V{v}: expected exit 1 at S6 with no PNG or timing")
    else:
        try:
            verify_csv(prefixes[3].with_suffix(".stdout").read_text(), RUNS, 3)
            if any("H" not in result or "panorama_sha256" not in result for result in versions):
                raise ValueError("Missing H or panorama after successful status")
            if len({result["config"] for result in versions}) != 1 or not re.search(r"\bseed=16\b", versions[3]["config"]):
                raise ValueError("Config or RANSAC seed disagrees across versions")
            row["drift_px"] = [projection_drift(versions[v]["H"], versions[3]["H"], width, height)
                               for v in VERSIONS[:-1]]
            if max(row["drift_px"]) >= DRIFT_PX:
                raise ValueError(f"H drift exceeds < {DRIFT_PX} px: {row['drift_px']}")
            canvas = versions[3]["canvas"]
            if any(result["canvas"] != canvas for result in versions):
                raise ValueError("Canvas dimensions or offsets differ")
            w, h, ox, oy = canvas
            if not (0 <= ox <= w - width and 0 <= oy <= h - height):
                raise ValueError("Canvas does not contain input A at the reported offset")
            row["canvas"] = canvas
            row["expanded"] = w > width or h > height
            row["rgb_delta"] = [rgb_delta(outputs[v], outputs[3]) for v in VERSIONS[:-1]]
            row["differs_from_p7_zero_rgb"] = any(row["rgb_delta"])
            if row["differs_from_p7_zero_rgb"]:
                print(f"{name}: RGB differs from the historical P7 value 0: {row['rgb_delta']}")
            if max(row["rgb_delta"]) > RGB_TOLERANCE:
                raise ValueError(f"Max RGB exceeds {RGB_TOLERANCE}: {row['rgb_delta']}")
        except (OSError, ValueError) as error:
            errors.append(str(error))
    ROWS.append(row)
    write_json(EVIDENCE / "cases" / f"{site}_{size}.json", row)
    print(name, statuses, errors or "PASS")
GATES["real_matrix"] = len(ROWS) == 12 and all(not row["errors"] for row in ROWS)
GATES["expanded_canvas_d2h"] = any(row.get("expanded") and not row["errors"] for row in ROWS)

cli = [sys.executable, "scripts/verify_v3_cli.py", "--stitch", STITCH, "--dataset", DATASET,
       "--out", EVIDENCE / "cli", "--require-t4"]
GATES["cli_csv"] = recorded("cli", cli, 1800)[0]["returncode"] == 0
bench = [sys.executable, "scripts/run_benchmark.py", "--stitch", STITCH, "--dataset", DATASET,
         "--out", EVIDENCE / "benchmark_contract", "--versions", "3", "--runs", "2", "--include-challenge"]
bench_result = recorded("benchmark_contract", bench, 3600)[0]
GATES["benchmark_runner"] = False
if bench_result["returncode"] == 0:
    root = EVIDENCE / "benchmark_contract/v3"
    with (root / "failures.csv").open(newline="") as stream:
        failures = {(r["site"], r["size"], r["exit_code"], r["error"]) for r in csv.DictReader(stream)}
    GATES["benchmark_runner"] = failures == {("challenge", size, "1", "NumericalFailure")
                                             for size in ("medium", "large")}
    if not GATES["benchmark_runner"]:
        print(f"Unexpected benchmark runner failures: {failures}")

## Challenge Gallery

Ở P7, challenge/small tạo ảnh nhưng ghép lệch. Hai mức medium và large trả NumericalFailure tại S6. Xem ảnh thực tế của V3 rồi ghi đánh giá hình học, kể cả khi mã thoát và RGB khớp baseline.

In [ ]:
# @title Inspect Challenge Status and Panoramas
from PIL import Image
import matplotlib.pyplot as plt

for row in ROWS:
    if row["case"].startswith("challenge/"):
        print(row["case"], row["statuses"], row["errors"] or "Status gate passed")
paths = [EVIDENCE / "panoramas/challenge/small" / f"v{v}.png" for v in VERSIONS]
if all(path.is_file() for path in paths):
    figure, axes = plt.subplots(1, 4, figsize=(19, 5))
    for version, (axis, path) in enumerate(zip(axes, paths)):
        with Image.open(path) as panorama:
            panorama.thumbnail((900, 600))
            axis.imshow(panorama)
        axis.set_title(f"V{version}")
        axis.axis("off")
    plt.show()

## Compute Sanitizer

Chạy memcheck cho suite V3 với fixture nhúng và CLI production của site1/small. Lượt ma trận đầy đủ ở trên sử dụng sáu fixture chính thức và 12 cấu hình ảnh thật. Hai lượt sanitizer phải có exit 0, dòng tổng kết lỗi và dòng tổng kết rò rỉ đầy đủ. Log gốc được giữ trong gói.

In [ ]:
# @title Run Memcheck for the Suite and Production CLI
from verify_v2_dataset import SANITIZER, memcheck_result
MEMCHECK = {}
for label, argv in {
    "suite": [SUITE],
    "cli": stitch_command(STITCH, *CASES[0][-2:], EVIDENCE / "memcheck_site1_small.png", 3, 1, True),
}.items():
    if not Path(argv[0]).is_file():
        MEMCHECK[label] = {"passed": False, "blocked": "Executable unavailable"}
        continue
    result, stdout, stderr = recorded(f"memcheck_{label}", [*SANITIZER, *argv], 1800)
    MEMCHECK[label] = memcheck_result(result, stdout, stderr)
    print(label, MEMCHECK[label])
GATES["memcheck"] = len(MEMCHECK) == 2 and all(result["passed"] for result in MEMCHECK.values())
write_json(EVIDENCE / "memcheck.json", MEMCHECK)

## Report and Evidence Export

Điền nhận xét sau khi xem gallery. Chạy lại ô tổng hợp để đóng gói kết quả, kể cả khi có lỗi. Gói giữ toàn bộ log và ảnh challenge, cùng ảnh site1/small từ lượt kiểm tra CLI. Các panorama chuẩn còn lại có hash trong JSON từng ca. Các JPEG của benchmark runner dùng trong lượt kiểm tra contract nằm ngoài ZIP.

In [ ]:
# @title Summarize and Download Evidence { display-mode: "form" }
CHALLENGE_SMALL_REVIEW = ""  # @param {type:"string"}
from google.colab import files

gates = globals().get("GATES", {})
rows = globals().get("ROWS", [])
synthetic_results = globals().get("SYNTHETIC_RESULTS", {})
memcheck = globals().get("MEMCHECK", {})
expected_gates = ("production_build", "v0_regression", "pipeline_regression", "six_synthetic",
                  "real_matrix", "expanded_canvas_d2h", "cli_csv", "benchmark_runner", "memcheck")
statuses = {key: ("PASS" if gates[key] else "FAIL") if key in gates else "BLOCKED" for key in expected_gates}
challenge_path = EVIDENCE / "panoramas/challenge/small/v3.png"
statuses["challenge_small_visual_review"] = (
    "PASS" if CHALLENGE_SMALL_REVIEW.strip() and challenge_path.is_file() else "BLOCKED"
)
statuses["source_unchanged"] = (
    "PASS" if source_snapshot()["snapshot_sha256"] == ENV["git"]["source_snapshot"] else "FAIL"
)
verdict = ("NOT READY — FAILED CHECKS" if "FAIL" in statuses.values() else
           "BLOCKED — MISSING REQUIRED EVIDENCE" if "BLOCKED" in statuses.values() else
           "READY FOR REVIEW")
write_json(EVIDENCE / "verification_summary.json", {"gates": statuses, "cases": rows,
           "synthetic": synthetic_results, "challenge_small_review": CHALLENGE_SMALL_REVIEW,
           "memcheck": memcheck, "environment": ENV, "verdict": verdict})

report = ["# P8-T04 — Colab T4 Run", "", f"- Production SHA: {PRODUCTION_SHA}",
          f"- Verification SHA: {VERIFY_SHA}", f"- GPU: {ENV['gpu_details']}", "",
          "## Synthetic verification", "", "| Case | Result |", "|---|---|"]
report.extend(f"| {key} | {value} |" for key, value in sorted(synthetic_results.items()))
report += ["", "## Real-image regression", "", "| Case | Status V0/V1/V2/V3 | Drift vs V0/V1/V2 (px) | Canvas | RGB delta | Findings |",
           "|---|---|---|---|---|---|"]
for row in rows:
    report.append(f"| {row['case']} | {' / '.join(row['statuses'])} | {row.get('drift_px', '—')} | "
                  f"{row.get('canvas', '—')} | {row.get('rgb_delta', '—')} | {'; '.join(row['errors']) or '—'} |")
report += ["", "## Challenge visual review", "", CHALLENGE_SMALL_REVIEW or "Pending visual review.",
           "", "## CLI and Compute Sanitizer", "", f"- T02 CSV: {T02_CSV_HEADER}",
           "- Raw CLI evidence: cli/ and logs/cli.*", "- Raw sanitizer evidence: logs/memcheck_*",
           "", "## Acceptance gates", "", f"**{verdict}**", ""]
for name, result in memcheck.items():
    report.append(f"- Memcheck {name}, exit {result.get('returncode')}: "
                  + (" / ".join(result.get("lines", [])) or result.get("blocked", "Missing summaries")))
differences = [row["case"] for row in rows if row.get("differs_from_p7_zero_rgb")]
if differences:
    report.extend(["", "RGB differences from the historical P7 value zero: " + ", ".join(differences)])
report.extend(f"- {key}: {value}" for key, value in statuses.items())
report_text = "\n".join(report) + "\n"
(EVIDENCE / "p8-v3-verification-run.md").write_text(report_text, encoding="utf-8")
display(Markdown(report_text))
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
archive_path = WORK / f"Group16_V3_P8-T04_TeslaT4_Evidence_{VERIFY_SHA[:8]}_{stamp}.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EVIDENCE.rglob("*")):
        if not path.is_file():
            continue
        rel = path.relative_to(EVIDENCE)
        if (rel.parts[0] == "panoramas" and rel.parts[1] != "challenge") or rel.parts[0] == "benchmark_contract" and path.suffix == ".jpg":
            continue
        archive.write(path, rel.as_posix())
checksum = archive_path.with_suffix(".zip.sha256")
checksum.write_text(f"{sha256(archive_path)}  {archive_path.name}\n", encoding="utf-8")
print(checksum.read_text())
files.download(str(archive_path))
files.download(str(checksum))

## References and Scope

- [Báo cáo nghiệm thu P7](docs/report/p7-v2-verification.md)
- [Báo cáo và trạng thái P8-T04](docs/report/p8-v3-verification.md)
- [Lệnh build và kiểm thử của nhóm](tests/README.md)